# Breast Cancer (VinDr-Mammo) — Qwen2.5-VL Models A & B

Pipeline:
1. Paths + load CSVs
2. Grouped labels (top findings + Other)
3. Train/val/test splits (VinDr official + carved val)
4. Box scaling helper (PNG-scaled, normalized 0–1000)
5. Build instruction JSONLs (plain + grounded)
6. YOLO grouped data prep
7. Train Qwen2.5-VL-7B Model A (plain grouped)
8. Train Qwen2.5-VL-7B Model B (grounded grouped)
9. Inference
10. Evaluation

All runs in-notebook on H200.

## 1. Setup & paths

In [ ]:
import os, json, ast, re, random, shutil
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import torch

DATA_ROOT = Path(os.environ.get('VINDR_ROOT', 'data/vindr_mammo'))  # see README -> Data
RAW       = DATA_ROOT / 'vindr-mammo-a-large-scale-benchmark-dataset-for-computer-aided-detection-and-diagnosis-in-full-field-digital-mammography-1.0.0'
PNG       = DATA_ROOT / 'png'
ANN       = DATA_ROOT / 'annotations_processed'
LORA_DATA = DATA_ROOT / 'lora_data'
LORA_RUNS = DATA_ROOT / 'lora_runs'
YOLO_DIR  = DATA_ROOT / 'yolo'
RUNS      = DATA_ROOT / 'runs'

for d in [ANN, LORA_DATA, LORA_RUNS, YOLO_DIR, RUNS]:
    d.mkdir(parents=True, exist_ok=True)

BREAST_CSV  = RAW / 'breast-level_annotations.csv'
FINDING_CSV = RAW / 'finding_annotations.csv'

breast_df  = pd.read_csv(BREAST_CSV)
finding_df = pd.read_csv(FINDING_CSV)

# dropped one corrupt image (failed DICOM->PNG conversion)
BAD_IDS = {'3e71b089f6e55d5791771b05f88518dd'}
breast_df  = breast_df[~breast_df['image_id'].isin(BAD_IDS)].reset_index(drop=True)
finding_df = finding_df[~finding_df['image_id'].isin(BAD_IDS)].reset_index(drop=True)

print('Breast rows:', len(breast_df), ' Finding rows:', len(finding_df))
print('PNG files:', len(list(PNG.glob('*.png'))))
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU')

Breast rows: 19999  Finding rows: 20485
PNG files: 19999
CUDA: True | NVIDIA H200 NVL


## 2. Grouped labels

multi-label, with rare findings collapsed into clinically meaningful groups:

| Group | Includes | Why |
|---|---|---|
| Mass | Mass | 1,123+ |
| Suspicious Calcification | Suspicious Calcification | 402+ |
| Asymmetry | Asymmetry, Focal Asymmetry, Global Asymmetry | clinically related, individually small |
| Architectural Distortion | Architectural Distortion | important cancer sign |
| Suspicious Lymph Node | Suspicious Lymph Node | clinically meaningful |
| Other Suspicious Signs | Skin Thickening, Skin Retraction, Nipple Retraction | too rare individually |

In [ ]:
GROUPS = {
    'Mass':                     ['Mass'],
    'Suspicious Calcification': ['Suspicious Calcification'],
    'Asymmetry':                ['Asymmetry', 'Focal Asymmetry', 'Global Asymmetry'],
    'Architectural Distortion': ['Architectural Distortion'],
    'Suspicious Lymph Node':    ['Suspicious Lymph Node'],
    'Other Suspicious Signs':   ['Skin Thickening', 'Skin Retraction', 'Nipple Retraction'],
}

GROUPED_CLASSES = list(GROUPS.keys())

GROUP_TO_PHRASE = {
    'Mass': 'A suspicious mass is present',
    'Suspicious Calcification': 'Suspicious calcifications are present',
    'Asymmetry': 'An asymmetry is present',
    'Architectural Distortion': 'Architectural distortion is present',
    'Suspicious Lymph Node': 'A suspicious lymph node is present',
    'Other Suspicious Signs': 'Other suspicious mammographic signs are present',
}

GROUP_TO_KNOWLEDGE = {
    'Mass': 'A mammographic mass is a space-occupying lesion with a visible shape, margin, and density.',
    'Suspicious Calcification': 'Suspicious calcifications appear as small high-density bright foci, often grouped or clustered.',
    'Asymmetry': 'Asymmetry is an area of denser breast tissue without a clear mass, possibly focal or global.',
    'Architectural Distortion': 'Architectural distortion appears as pulling or disruption of normal breast tissue without a definite mass.',
    'Suspicious Lymph Node': 'A suspicious lymph node appears enlarged, dense, rounded, or with abnormal morphology.',
    'Other Suspicious Signs': 'Other suspicious signs include skin thickening, nipple retraction, or skin retraction.',
}

# fix inconsistencies in the finding_categories col of finding_df
def parse_finding_categories(x):
    if pd.isna(x): return []
    if isinstance(x, list): return x
    try:
        # sometimes the col is a string representation of a list; ast.literal_eval turns it back
        v = ast.literal_eval(str(x))
        # if list, return it; if it's a single value, turn it into a string wrapped in a list
        return v if isinstance(v, list) else [str(v)]
    # safety net so code doesn't break if the string can't be parsed; just returns it as a single-item list
    except Exception:
        return [str(x)]

# takes clean finding list and maps it to the classes defined in GROUPS, returning a sorted list of unique groups
def map_to_groups(categories):
    cats = parse_finding_categories(categories)
    if 'No Finding' in cats:
        return []
    # sets can't hold duplicates; ensures that if a pt has multiple findings in the same group, it's only recorded once
    out = set()
    # loops through GROUPS (where g = group name, raws = list of raw findings) to check if any raw findings are in cats; if so, adds group to the output set
    for g, raws in GROUPS.items():
        for r in raws:
            if r in cats:
                out.add(g)
    return sorted(out) # sorts alphabetically 

# maps each image_id to a set of corresponding group labels using map_to_groups 
# tells WHAT is in the image -- used for global image classification/filtering
image_to_groups = defaultdict(set)
# loops through each row, passes row's categories to map_to_groups, and adds the result to the image's set
for _, r in finding_df.iterrows():
    for g in map_to_groups(r['finding_categories']):
        image_to_groups[r['image_id']].add(g)
# converts sets to sorted lists. result is a dict structured like {'image_01': ['Mass', 'Asymmetry'], ...}
image_to_groups = {k: sorted(v) for k, v in image_to_groups.items()}

"""
parse_finding_categories / map_to_groups / image_to_groups: build a quick-reference dictionary where you can look up any image_id and get a sorted, unique list of the high-level categories present in that image
"""

counts = Counter()
for groups in image_to_groups.values():
    for g in groups:
        counts[g] += 1

n_normal = len(breast_df) - sum(1 for img in breast_df['image_id'] if img in image_to_groups and len(image_to_groups[img]) > 0)
print(f'{"Normal (no finding)":30s} {n_normal}')
for g in GROUPED_CLASSES:
    print(f'{g:30s} {counts[g]}')

Normal (no finding)            18231
Mass                           1113
Suspicious Calcification       442
Asymmetry                      390
Architectural Distortion       119
Suspicious Lymph Node          53
Other Suspicious Signs         86


## 3. Splits

VinDr-Mammo provides an official `split` column (`training` / `test`). we create a validation set from the training studies, keeping all views of a study together (study-level split). 

important that one pt's images don't get split up bc model could "cheat" by having already seen that pt's anatomy

In [ ]:
SEED = 0
VAL_FRAC = 0.10

train_studies_all = sorted(breast_df.loc[breast_df['split'] == 'training', 'study_id'].unique().tolist())
test_studies      = sorted(breast_df.loc[breast_df['split'] == 'test',     'study_id'].unique().tolist())

# carves out a validation set from the training studies
rng = random.Random(SEED)
rng.shuffle(train_studies_all)
n_val = int(len(train_studies_all) * VAL_FRAC)
val_studies   = sorted(train_studies_all[:n_val])
train_studies = sorted(train_studies_all[n_val:])

# returns all image_ids corresponding to the given study_ids
def studies_to_images(study_ids):
    return sorted(breast_df.loc[breast_df['study_id'].isin(study_ids), 'image_id'].unique().tolist())

train_ids = studies_to_images(train_studies)
val_ids   = studies_to_images(val_studies)
test_ids  = studies_to_images(test_studies)

print(f'Train studies: {len(train_studies):5d}  images: {len(train_ids):5d}')
print(f'Val   studies: {len(val_studies):5d}  images: {len(val_ids):5d}')
print(f'Test  studies: {len(test_studies):5d}  images: {len(test_ids):5d}')

# guarantee no study overlap
assert not (set(train_studies) & set(val_studies))
assert not (set(train_studies) & set(test_studies))
assert not (set(val_studies)   & set(test_studies))

# save splits
(ANN / 'splits.json').write_text(json.dumps({
    'train_studies': train_studies, 'val_studies': val_studies, 'test_studies': test_studies,
    'train_ids': train_ids, 'val_ids': val_ids, 'test_ids': test_ids,
}))
print('Saved splits to', ANN / 'splits.json')

Train studies:  3600  images: 14399
Val   studies:   400  images:  1600
Test  studies:  1000  images:  4000
Saved splits to $VINDR_ROOT/annotations_processed/splits.json


## 4. Box scaling

annotations store boxes in **original DICOM** coordinates. PNGs may have different dimensions. we scale to PNG pixel coords, then normalize to 0–1000 for the LLM target text (standard for Qwen2.5-VL grounding)

In [ ]:
_png_dim_cache = {}

# get dim of png file: if already cached return. if not, get size + cache it for future quick access
def get_png_size(image_id):
    if image_id in _png_dim_cache:
        return _png_dim_cache[image_id]
    p = PNG / f'{image_id}.png'
    if not p.exists():
        return None
    with Image.open(p) as im:
        _png_dim_cache[image_id] = im.size  # (w, h)
    return _png_dim_cache[image_id]

def box_to_norm1000(row, image_id):
    """Original DICOM coords -> PNG-scaled -> normalized 0-1000.
    Returns [x1, y1, x2, y2] ints, or None if invalid."""
    size = get_png_size(image_id)
    if size is None: return None
    png_w, png_h = size

    orig_w, orig_h = float(row['width']), float(row['height'])
    sx, sy = png_w / orig_w, png_h / orig_h

    x1 = float(row['xmin']) * sx
    y1 = float(row['ymin']) * sy
    x2 = float(row['xmax']) * sx
    y2 = float(row['ymax']) * sy

    x1 = max(0, min(png_w, x1)); x2 = max(0, min(png_w, x2))
    y1 = max(0, min(png_h, y1)); y2 = max(0, min(png_h, y2))
    if x2 - x1 < 2 or y2 - y1 < 2:
        return None

    return [
        int(round(x1 / png_w * 1000)),
        int(round(y1 / png_h * 1000)),
        int(round(x2 / png_w * 1000)),
        int(round(y2 / png_h * 1000)),
    ]

# build image_id -> list of {group, box} with PNG-scaled normalized coords
# tells WHAT type of finding it is (group) and WHERE each finding is (box)
image_to_boxes = defaultdict(list)
# loops through each row of finding_df, skipping rows with missing coords. 
for _, r in tqdm(finding_df.iterrows(), total=len(finding_df), desc='Scaling boxes'):
    if pd.isna(r.get('xmin', np.nan)): continue
    # maps finding cats to groups and normalizes the bounding box coords
    groups = map_to_groups(r['finding_categories'])
    if not groups: continue
    box = box_to_norm1000(r, r['image_id'])
    if box is None: continue
    # appends group and box to corresponding image_id in image_to_boxes
    for g in groups:
        image_to_boxes[r['image_id']].append({'group': g, 'box': box})

print(f'Images with at least one grouped box: {len(image_to_boxes)}')

# check
sample_id = next(iter(image_to_boxes))
print(f'\nSample {sample_id}: {image_to_boxes[sample_id]}')

Images with at least one grouped box: 1768

Sample <image_id>: [{'group': 'Mass', 'box': [841, 492, 887, 527]}]


## 5. Build instruction JSONLs

multi-task instruction format (RadVLM-inspired). each image yields multiple examples:
1. **Finding classification** — what findings are present?
2. **BI-RADS + density** — what is the BI-RADS category and breast density?
3. **Report** — plain or grounded, depending on tier
4. **Per-finding grounding** (grounded tier only) — locate each finding with knowledge prompt

two tiers:
- `plain_grouped` — no boxes (Model A)
- `grounded_grouped` — boxes as `<box>x1 y1 x2 y2</box>` in 0–1000 normalized coords (Model B)

In [ ]:
PLAIN_DIR = LORA_DATA / 'plain_grouped'
GND_DIR   = LORA_DATA / 'grounded_grouped'
PLAIN_DIR.mkdir(parents=True, exist_ok=True)
GND_DIR.mkdir(parents=True, exist_ok=True)

SYSTEM_MAMMO = (
    'You are an expert radiology assistant for mammography. '
    'Answer carefully using only the image information.'
)

# gets info for a given image_id from breast_df
def get_breast_row(image_id):
    rows = breast_df[breast_df['image_id'] == image_id]
    return rows.iloc[0] if len(rows) else None

# openai chat completion API , modified for multi-modal inputs
def make_messages(image_path, user_text, assistant_text):
    return [
        {'role': 'system',    'content': [{'type': 'text', 'text': SYSTEM_MAMMO}]},
        # user input can be text and/or image
        {'role': 'user',      'content': [{'type': 'image', 'image': str(image_path)}, 
                                          {'type': 'text', 'text': user_text}]},
        {'role': 'assistant', 'content': [{'type': 'text', 'text': assistant_text}]},
    ]

'''----------------task isolation-------------------'''

# checks if groups is empty. if not, lists the suspicious findings present
# only cares about the localized physical findings
def finding_classification_answer(groups):
    if not groups: return 'No suspicious mammographic finding is present.'
    return 'The suspicious mammographic findings present are: ' + ', '.join(groups) + '.'

# gets BI-RADS cat and density from the breast_df row
def birads_density_answer(row):
    return f'The BI-RADS category is {row["breast_birads"]}. The breast density is {row["breast_density"]}.'

# combines local findings and global metadata into one narrative paragraph
def plain_report_answer(groups, row):
    parts = [birads_density_answer(row)]
    if not groups:
        parts.append('No suspicious mammographic finding is present.')
    else:
        for g in groups:
            parts.append(GROUP_TO_PHRASE[g] + '.')
    return ' '.join(parts)

# adds in box coords
def grounded_report_answer(image_id, groups, row):
    parts = [birads_density_answer(row)]
    items = image_to_boxes.get(image_id, [])
    if not groups or not items:
        parts.append('No suspicious mammographic finding is present.')
        return ' '.join(parts)
    for it in items:
        b = it['box']
        parts.append(f'{GROUP_TO_PHRASE[it["group"]]} <box>{b[0]} {b[1]} {b[2]} {b[3]}</box>.')
    return ' '.join(parts)

# formats one image_id into a list of multiple different training examples
# grounding can be toggled off/on for models A/B
def build_examples_for_image(image_id, grounded=False):
    png_path = PNG / f'{image_id}.png'
    if not png_path.exists(): return []
    row = get_breast_row(image_id)
    if row is None: return []

    groups = image_to_groups.get(image_id, [])
    examples = []

    # Task 1: finding classification
    examples.append({
        'image': str(png_path), 'task': 'finding_classification',
        'messages': make_messages(png_path,
            'What suspicious mammographic findings are present?',
            finding_classification_answer(groups))
    })

    # Task 2: BI-RADS + density
    examples.append({
        'image': str(png_path), 'task': 'birads_density',
        'messages': make_messages(png_path,
            'What is the BI-RADS category and breast density?',
            birads_density_answer(row))
    })

    # Task 3: report
    if grounded:
        report = grounded_report_answer(image_id, groups, row)
        user = ('Generate a grounded mammography report. For each suspicious finding, '
                'append its location as <box>x_min y_min x_max y_max</box> in normalized 0-1000 coordinates.')
        task = 'grounded_report'
    else:
        report = plain_report_answer(groups, row)
        user = 'Generate a concise mammography finding report.'
        task = 'plain_report'
    examples.append({'image': str(png_path), 'task': task,
                     'messages': make_messages(png_path, user, report)})

    # Task 4: per-finding grounding (grounded only)
    if grounded:
        for it in image_to_boxes.get(image_id, []):
            g, b = it['group'], it['box']
            examples.append({
                'image': str(png_path), 'task': 'single_finding_grounding',
                'messages': make_messages(png_path,
                    f'Locate {g}. {GROUP_TO_KNOWLEDGE[g]}',
                    f'The {g} is located at <box>{b[0]} {b[1]} {b[2]} {b[3]}</box>.')
            })

    return examples

# for the specified split, iterates through the image_ids, builds examples for each, and writes them to a jsonl file. 
def write_split(ids, split_name, out_dir, grounded):
    out_path = out_dir / f'{split_name}.jsonl'
    n_img = n_ex = 0 # keeps track of num imgs and exs written to file
    with open(out_path, 'w') as f:
        for image_id in tqdm(sorted(ids), desc=f'{out_dir.name}/{split_name}'):
            exs = build_examples_for_image(image_id, grounded=grounded)
            if not exs: continue
            n_img += 1
            for ex in exs:
                f.write(json.dumps(ex) + '\n')
                n_ex += 1
    print(f'  {split_name}: images={n_img}, examples={n_ex}')


# execution blocks: loop through each split for plain and grounded reports
print('=== Plain grouped (Model A) ===')
for split, ids in [('train', train_ids), ('val', val_ids), ('test', test_ids)]:
    write_split(ids, split, PLAIN_DIR, grounded=False)

print('\n=== Grounded grouped (Model B) ===')
for split, ids in [('train', train_ids), ('val', val_ids), ('test', test_ids)]:
    write_split(ids, split, GND_DIR, grounded=True)

=== Plain grouped (Model A) ===


  train: images=14399, examples=43197


  val: images=1600, examples=4800


  test: images=4000, examples=12000

=== Grounded grouped (Model B) ===


  train: images=14399, examples=44922


  val: images=1600, examples=5022


  test: images=4000, examples=12482


## 6. YOLO grouped data prep

detector baseline with correct box scaling (DICOM coords → PNG coords → YOLO normalized [0,1]).

In [ ]:
YOLO_OUT = YOLO_DIR / 'grouped'
# maps each group to a unique integer ID for YOLO training
YOLO_GROUP_TO_ID = {c: i for i, c in enumerate(GROUPED_CLASSES)}

def write_yolo_split(ids, split_name):
    img_dir = YOLO_OUT / split_name / 'images'
    lbl_dir = YOLO_OUT / split_name / 'labels'
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)

    n_img = n_box = 0
    for image_id in tqdm(sorted(ids), desc=f'YOLO {split_name}'):
        src = PNG / f'{image_id}.png'
        if not src.exists(): continue

        size = get_png_size(image_id)
        if size is None: continue
        W, H = size

        dst_img = img_dir / f'{image_id}.png'
        if not dst_img.exists():
            os.symlink(src, dst_img)   # symlink instead of copy (saves disk)

        # raw bounding box (DICOM) -> PNG space -> YOLO format (norm center x, center y, w, h)
        lines = []
        for _, r in finding_df[finding_df['image_id'] == image_id].iterrows():
            if pd.isna(r.get('xmin', np.nan)): continue
            groups = map_to_groups(r['finding_categories'])
            if not groups: continue

            orig_w, orig_h = float(r['width']), float(r['height'])
            sx, sy = W / orig_w, H / orig_h
            x1 = max(0, min(W, float(r['xmin']) * sx))
            x2 = max(0, min(W, float(r['xmax']) * sx))
            y1 = max(0, min(H, float(r['ymin']) * sy))
            y2 = max(0, min(H, float(r['ymax']) * sy))
            bw, bh = x2 - x1, y2 - y1
            if bw <= 1 or bh <= 1: continue

            xc = ((x1 + x2) / 2) / W
            yc = ((y1 + y2) / 2) / H
            bw, bh = bw / W, bh / H

            for g in groups:
                lines.append(f'{YOLO_GROUP_TO_ID[g]} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}')
                n_box += 1

        (lbl_dir / f'{image_id}.txt').write_text('\n'.join(lines))
        n_img += 1

    print(f'  {split_name}: images={n_img}, boxes={n_box}')

for split, ids in [('train', train_ids), ('val', val_ids), ('test', test_ids)]:
    write_yolo_split(ids, split)

# YOLOv8-style YAML file for training
yaml = (
    f'path: {YOLO_OUT}\n'
    f'train: train/images\n'
    f'val: val/images\n'
    f'test: test/images\n'
    f'nc: {len(GROUPED_CLASSES)}\n'
    f'names: {GROUPED_CLASSES}\n'
)
yaml_path = DATA_ROOT / 'mammo_grouped.yaml'
yaml_path.write_text(yaml)
print('YAML written:', yaml_path)

  train: images=14399, boxes=1725


  val: images=1600, boxes=222


  test: images=4000, boxes=482
YAML written: $VINDR_ROOT/mammo_grouped.yaml


In [ ]:
from ultralytics import YOLO

# smaller, more stable model + medical-appropriate hyperparams
yolo_model = YOLO('yolo11s.pt')  # downgrade from 'l' to 's' for stability

yolo_model.train(
    data=str(DATA_ROOT / 'mammo_grouped.yaml'),
    imgsz=1024,
    epochs=80,                # bumped — slower lr means longer training
    batch=8,
    seed=0,
    project=str(RUNS),
    name='mammo_yolo11s_grouped_v2',
    device=0,
    optimizer='AdamW',
    lr0=0.0005,               # halved from 0.001 — was causing NaN
    cos_lr=True,
    patience=30,              # more patience — medical detection learns slowly
    exist_ok=True,
    # ***
    mosaic=0.0,               # NO mixing patients
    mixup=0.0,                # NO mixing patients
    copy_paste=0.0,           # NO copy-paste
    fliplr=0.0,               # laterality distinction
    flipud=0.0,
    degrees=5.0,              # small rotation only
    translate=0.05,
    scale=0.1,
    shear=0.0,
    perspective=0.0,
    hsv_h=0.0,                # grayscale, no hue
    hsv_s=0.0,                # grayscale, no saturation
    hsv_v=0.2,                # brightness OK
    # Numerical stability:
    amp=True,                 # default; if NaN persists, set amp=False
    close_mosaic=0,           # disable the mosaic-close hack (no mosaic anyway)
)

Ultralytics 8.4.60 🚀 Python-3.11.15 torch-2.12.0+cu130 CUDA:0 (NVIDIA H200 NVL, 143156MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=0, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=$VINDR_ROOT/mammo_grouped.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.2, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=mammo_yolo11s_grouped_v2, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_m

<site-packages>/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 8, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
<site-packages>/torch/utils/data/dataloader.py:430: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 8, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


optimizer: AdamW(lr=0.0005, momentum=0.937) with parameter groups 81 weight(decay=0.0), 88 weight(decay=0.0005), 87 bias(decay=0.0)
Plotting labels to $VINDR_ROOT/runs/mammo_yolo11s_grouped_v2/labels.jpg... 
Image sizes 1024 train, 1024 val
Using 8 dataloader workers
Logging results to $VINDR_ROOT/runs/mammo_yolo11s_grouped_v2
Starting training for 80 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       1/80      39.5G      1.336      7.331      1.148          7       1024: 100% ━━━━━━━━━━━━ 1800/1800 16.2it/s 1:51<0.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 100/100 22.8it/s 4.4s.1s
                   all       1600        222   0.000848     0.0783    0.00144   0.000554

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/80      39.5G      1.249      2.525      1.123          7       1024: 100% ━━━━━━━━━━━━ 1800/1800 18.2it/s 1:39<0.1ss
   

In [ ]:
# loads in the VLM grounded results
preds_grounded = [json.loads(l) for l in open(LORA_RUNS / 'preds_grounded_grouped.jsonl')]
report_preds = [p for p in preds_grounded if p['task'] == 'grounded_report']
print(f'{len(report_preds)} grounded report predictions loaded')

4000 grounded report predictions loaded


In [ ]:
from ultralytics import YOLO

# load trained YOLO once (use best.pt from your training run)
# best.pt is the checkpoint with the best validation performance, which is what you want for inference
YOLO_CKPT = RUNS / 'mammo_yolo11l_grouped' / 'weights' / 'best.pt'
yolo_infer = YOLO(str(YOLO_CKPT))
print(f'Loaded YOLO from {YOLO_CKPT}')

# reverse map: YOLO class id -> group name
YOLO_ID_TO_GROUP = {i: c for i, c in enumerate(GROUPED_CLASSES)}

# passes image to YOLO model and returns a list of detected boxes with  group labels and confidence scores
def yolo_predict_boxes(image_id, conf_threshold=0.25):
    """Run YOLO on a PNG, return list of (group_label, [x1,y1,x2,y2], confidence)."""
    img_path = PNG / f'{image_id}.png'
    results = yolo_infer.predict(str(img_path), conf=conf_threshold, verbose=False)
    out = []
    # converts YOLO results to standard python formats 
    for r in results:
        for box in r.boxes:
            cls_id = int(box.cls[0].item())
            conf = float(box.conf[0].item())
            xyxy = box.xyxy[0].tolist()  # pixel coords [x1,y1,x2,y2]
            out.append((YOLO_ID_TO_GROUP.get(cls_id, 'Unknown'), xyxy, conf))
    return out

# creates diagnostic dashboard showing ground truth, grounded vlm, and YOLO side by side for a given image_id
def visualize_three_panel(image_id, pred_text, conf_threshold=0.25, figsize=(18, 10)):
    """GT | Model B | YOLO side-by-side."""
    img_path = PNG / f'{image_id}.png'
    img = Image.open(img_path)
    W, H = img.size

    fig, axes = plt.subplots(1, 3, figsize=figsize)

    # ---- Panel 1: GT ----
    axes[0].imshow(img, cmap='gray')
    axes[0].set_title(f'Ground Truth\n{image_id}', fontsize=11)
    for item in image_to_boxes.get(image_id, []):
        g, b = item['group'], item['box']
        x1, y1, x2, y2 = denorm_box(b, W, H)
        color = GROUP_COLORS.get(g, 'white')
        axes[0].add_patch(Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2,
                                    edgecolor=color, facecolor='none'))
        axes[0].text(x1, y1-10, g, color=color, fontsize=8,
                     bbox=dict(facecolor='black', alpha=0.7, pad=2))
    axes[0].axis('off')

    # ---- Panel 2: Model B ----
    axes[1].imshow(img, cmap='gray')
    axes[1].set_title('Model B (Grounded VLM)', fontsize=11)
    for g, b in parse_box_with_label(pred_text):
        x1, y1, x2, y2 = denorm_box(b, W, H)
        color = GROUP_COLORS.get(g, 'white')
        axes[1].add_patch(Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2,
                                    edgecolor=color, facecolor='none', linestyle='--'))
        axes[1].text(x1, y1-10, g, color=color, fontsize=8,
                     bbox=dict(facecolor='black', alpha=0.7, pad=2))
    axes[1].axis('off')

    # ---- Panel 3: YOLO ----
    axes[2].imshow(img, cmap='gray')
    axes[2].set_title(f'YOLO (conf ≥ {conf_threshold})', fontsize=11)
    for g, xyxy, conf in yolo_predict_boxes(image_id, conf_threshold=conf_threshold):
        x1, y1, x2, y2 = xyxy
        color = GROUP_COLORS.get(g, 'white')
        axes[2].add_patch(Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2,
                                    edgecolor=color, facecolor='none', linestyle=':'))
        axes[2].text(x1, y1-10, f'{g} ({conf:.2f})', color=color, fontsize=8,
                     bbox=dict(facecolor='black', alpha=0.7, pad=2))
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()

# run on some test cases that have GT findings
shown = 0
for p in report_preds:
    image_id = Path(p['image']).stem
    if image_id in image_to_boxes and shown < 5:
        print(f'\n=== {image_id} ===')
        print(f'GT:    {p["gt"][:180]}')
        print(f'Pred:  {p["pred"][:180]}')
        visualize_three_panel(image_id, p['pred'])
        shown += 1

Loaded YOLO from $VINDR_ROOT/runs/mammo_yolo11l_grouped/weights/best.pt


In [ ]:
# visually checking that boxes align
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

sample_files = list((YOLO_DIR / 'grouped' / 'train' / 'labels').glob('*.txt'))[:5]
for lbl_file in sample_files:
    image_id = lbl_file.stem
    img_path = YOLO_DIR / 'grouped' / 'train' / 'images' / f'{image_id}.png'
    img = PILImage.open(img_path)
    W, H = img.size
    
    fig, ax = plt.subplots(figsize=(6, 8))
    ax.imshow(img, cmap='gray')
    
    for line in lbl_file.read_text().strip().split('\n'):
        if not line: continue
        cls_id, xc, yc, bw, bh = line.split()
        cls_id, xc, yc, bw, bh = int(cls_id), float(xc), float(yc), float(bw), float(bh)
        x1 = (xc - bw/2) * W
        y1 = (yc - bh/2) * H
        ax.add_patch(Rectangle((x1, y1), bw*W, bh*H, linewidth=2, 
                               edgecolor='red', facecolor='none'))
        ax.text(x1, y1-5, GROUPED_CLASSES[cls_id], color='red', fontsize=10,
                bbox=dict(facecolor='black', alpha=0.7))
    ax.set_title(f'{image_id}')
    ax.axis('off')
    plt.show()

## 7. Train Qwen2.5-VL-7B — Model A (plain grouped)

In [ ]:
#import sys
#!{sys.executable} -m pip install -U transformers accelerate peft trl bitsandbytes qwen-vl-utils[decord]


... [long training log trimmed for the repo] ...

36m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 89.2 MB/s  0:00:00
  Attempting uninstall: transformersm╺━━━━━━━━━━━━━━━━━━━ 2/4 [qwen-vl-utils]
    Found existing installation: transformers 5.9.0━━━━━━━━━━━ 2/4 [qwen-vl-utils]
    Uninstalling transformers-5.9.0:━━╺━━━━━━━━━ 3/4 [transformers]
      Successfully uninstalled transformers-5.9.0━━━━━━━━━ 3/4 [transformers]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [transformers] [transformers]

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: <cluster-path> -m pip install --upgrade pip


In [ ]:
import torch, json, gc
from pathlib import Path
from datasets import Dataset
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
from qwen_vl_utils import process_vision_info
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

MODEL_ID = 'Qwen/Qwen2.5-VL-7B-Instruct'

def load_jsonl(path):
    return Dataset.from_list([json.loads(l) for l in open(path)])

# ---- change for Model A vs B ----
TIER = 'plain_grouped'   # 'grounded_grouped'
# ---------------------------------------

DATA_DIR = LORA_DATA / TIER
OUT_DIR  = LORA_RUNS / f'qwen25vl7b_{TIER}'
OUT_DIR.mkdir(parents=True, exist_ok=True)

train_ds = load_jsonl(DATA_DIR / 'train.jsonl')
val_ds   = load_jsonl(DATA_DIR / 'val.jsonl')

print('Model:', MODEL_ID, '| Tier:', TIER)
print('Train:', len(train_ds), 'Val:', len(val_ds))
print('Sample assistant text:', train_ds[0]['messages'][-1]['content'][0]['text'][:200])

Model: Qwen/Qwen2.5-VL-7B-Instruct | Tier: plain_grouped
Train: 43197 Val: 4800
Sample assistant text: No suspicious mammographic finding is present.


In [ ]:
# load processor + model (bf16, no quant — H200 has the VRAM)
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map='auto',
)
model.config.use_cache = False
print('Loaded.', 'Params:', sum(p.numel() for p in model.parameters()) / 1e9, 'B')

Loaded. Params: 8.292166656 B


In [ ]:
# necessary bc None fields can break processor
def clean_messages(messages):
    """Remove None fields that HF Datasets adds during schema unification."""
    out = []
    for m in messages:
        new_content = []
        for c in m['content']:
            cleaned = {k: v for k, v in c.items() if v is not None}
            new_content.append(cleaned)
        out.append({'role': m['role'], 'content': new_content})
    return out

# combines multiple examples into a single batch for training, handling text, images, and videos
def collate_qwen(examples):
    messages = [clean_messages(ex['messages']) for ex in examples]
    texts = [processor.apply_chat_template(m, tokenize=False, add_generation_prompt=False) for m in messages]
    image_inputs, video_inputs = process_vision_info(messages)
    batch = processor(text=texts, images=image_inputs, videos=video_inputs,
                      padding=True, return_tensors='pt')
    labels = batch['input_ids'].clone()
    pad_id = processor.tokenizer.pad_token_id
    if pad_id is not None:
        labels[labels == pad_id] = -100
    image_token_id = processor.tokenizer.convert_tokens_to_ids('<|image_pad|>')
    if image_token_id is not None:
        labels[labels == image_token_id] = -100
    batch['labels'] = labels
    return batch

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
)

cfg = SFTConfig(
    output_dir=str(OUT_DIR),
    num_train_epochs=2,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8, # batch slicing: processes 1 image at a time, but sums up the mathematical loss over 8 sequential images before taking a single optimization step
    learning_rate=1e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy='epoch',
    eval_strategy='epoch',
    bf16=True,
    gradient_checkpointing=True,
    remove_unused_columns=False,
    report_to='none',
    dataset_kwargs={'skip_prepare_dataset': True},
)

trainer = SFTTrainer(
    model=model, args=cfg,
    train_dataset=train_ds, eval_dataset=val_ds,
    peft_config=lora,
    data_collator=collate_qwen,
    processing_class=processor,
)
trainer.train()
trainer.save_model(str(OUT_DIR))
print('Saved to', OUT_DIR)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[RANK 0] Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'bos_token_id': None, 'pad_token_id': 151643}.


IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



Saved to $VINDR_ROOT/lora_runs/qwen25vl7b_plain_grouped


## 9. Inference

In [ ]:
from peft import PeftModel

# ---- choose which model ----
EVAL_TIER = 'plain_grouped'   # 'grounded_grouped'
# ----------------------------

ADAPTER_DIR = LORA_RUNS / f'qwen25vl7b_{EVAL_TIER}'
EVAL_DATA   = LORA_DATA / EVAL_TIER / 'test.jsonl'
PRED_OUT    = LORA_RUNS / f'preds_{EVAL_TIER}.jsonl'

processor = AutoProcessor.from_pretrained(MODEL_ID)
base = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto')
model = PeftModel.from_pretrained(base, str(ADAPTER_DIR))
model.eval()
print('Loaded adapter:', ADAPTER_DIR)

Loaded adapter: $VINDR_ROOT/lora_runs/qwen25vl7b_plain_grouped


In [ ]:
@torch.no_grad()
# strip assistant's answer from each test example. @ inference model has never seen the ground-truth answer
def generate_one(messages, max_new_tokens=256):
    inf_messages = [m for m in messages if m['role'] != 'assistant']
    text = processor.apply_chat_template(inf_messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(inf_messages)
    inputs = processor(text=[text], images=image_inputs, videos=video_inputs,
                       padding=True, return_tensors='pt').to(model.device)
    gen = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    trimmed = gen[:, inputs['input_ids'].shape[1]:]
    return processor.batch_decode(trimmed, skip_special_tokens=True)[0].strip()

# generate predictions on test set
test_examples = [json.loads(l) for l in open(EVAL_DATA)]
print(f'{len(test_examples)} test examples')

with open(PRED_OUT, 'w') as f:
    for ex in tqdm(test_examples, desc='Generating'):
        pred = generate_one(ex['messages'])
        gt = ex['messages'][-1]['content'][0]['text']
        out = {
            'image': ex['image'],
            'task': ex['task'],
            'gt': gt,
            'pred': pred,
        }
        f.write(json.dumps(out) + '\n')

print('Saved predictions to', PRED_OUT)

12000 test examples


Saved predictions to $VINDR_ROOT/lora_runs/preds_plain_grouped.jsonl


## 10. Evaluation

per-task metrics:
- finding_classification -> multi-label F1 over grouped classes
- birads_density -> BI-RADS accuracy, density accuracy
- plain_report / grounded_report -> finding F1 from extracted phrases
- single_finding_grounding (Model B only) -> box IoU

In [ ]:
from sklearn.metrics import f1_score, precision_recall_fscore_support, accuracy_score, cohen_kappa_score

preds = [json.loads(l) for l in open(PRED_OUT)]
by_task = defaultdict(list)
for p in preds: by_task[p['task']].append(p)
for t, items in by_task.items():
    print(f'{t:30s} {len(items)}')

# ---- helpers ----
GROUP_KEYWORDS = {
    'Mass': ['mass'],
    'Suspicious Calcification': ['calcification'],
    'Asymmetry': ['asymmetry'],
    'Architectural Distortion': ['architectural distortion', 'distortion'],
    'Suspicious Lymph Node': ['lymph node'],
    'Other Suspicious Signs': ['other suspicious', 'skin thickening', 'skin retraction', 'nipple retraction'],
}

def extract_findings(text):
    t = text.lower()
    out = set()
    for g, kws in GROUP_KEYWORDS.items():
        if any(k in t for k in kws):
            out.add(g)
    return out

def to_binary_vec(groups):
    return [1 if g in groups else 0 for g in GROUPED_CLASSES]

BIRADS_RE  = re.compile(r'bi-?rads\s*(?:category\s*)?([1-5])', re.I)
DENSITY_RE = re.compile(r'density\s*([abcd])', re.I)
BOX_RE     = re.compile(r'<box>\s*(\d+)\s+(\d+)\s+(\d+)\s+(\d+)\s*</box>')

def extract_birads(text):
    m = BIRADS_RE.search(text)
    return m.group(1) if m else None

def extract_density(text):
    m = DENSITY_RE.search(text)
    return m.group(1).upper() if m else None

def extract_boxes(text):
    return [list(map(int, m)) for m in BOX_RE.findall(text)]

def iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)
    inter = iw * ih
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / ua if ua > 0 else 0.0

finding_classification         4000
birads_density                 4000
plain_report                   4000


In [ ]:
# finding classification metrics
if 'finding_classification' in by_task:
    items = by_task['finding_classification']
    y_true, y_pred = [], []
    for it in items:
        y_true.append(to_binary_vec(extract_findings(it['gt'])))
        y_pred.append(to_binary_vec(extract_findings(it['pred'])))
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    micro_f1 = f1_score(y_true, y_pred, average='micro', zero_division=0)
    print(f'\nFinding classification: macro F1 = {macro_f1:.3f}, micro F1 = {micro_f1:.3f}')
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average=None, zero_division=0, labels=list(range(len(GROUPED_CLASSES))))
    for i, g in enumerate(GROUPED_CLASSES):
        print(f'  {g:30s} P={p[i]:.3f}  R={r[i]:.3f}  F1={f[i]:.3f}')

# BI-RADS + density
if 'birads_density' in by_task:
    items = by_task['birads_density']
    bg, bp, dg, dp = [], [], [], []
    for it in items:
        b_g, b_p = extract_birads(it['gt']), extract_birads(it['pred'])
        d_g, d_p = extract_density(it['gt']), extract_density(it['pred'])
        if b_g and b_p: bg.append(int(b_g)); bp.append(int(b_p))
        if d_g and d_p: dg.append(d_g); dp.append(d_p)
    print(f'\nBI-RADS:  acc={accuracy_score(bg, bp):.3f}, weighted kappa={cohen_kappa_score(bg, bp, weights="quadratic"):.3f}  ({len(bg)} valid)')
    print(f'Density:  acc={accuracy_score(dg, dp):.3f}, weighted kappa={cohen_kappa_score(dg, dp, weights="quadratic"):.3f}  ({len(dg)} valid)')

# grounding IoU (model B / grounded tier only)
if 'single_finding_grounding' in by_task:
    items = by_task['single_finding_grounding']
    ious = []
    for it in items:
        gt_b = extract_boxes(it['gt'])
        pr_b = extract_boxes(it['pred'])
        if not gt_b or not pr_b:
            ious.append(0.0); continue
        ious.append(iou(gt_b[0], pr_b[0]))
    ious = np.array(ious)
    print(f'\nGrounding: mean IoU = {ious.mean():.3f}, IoU@0.5 = {(ious >= 0.5).mean():.3f}  ({len(ious)} cases)')

# report extraction (findings from generated report)
for task_name in ['plain_report', 'grounded_report']:
    if task_name not in by_task: continue
    items = by_task[task_name]
    y_true, y_pred = [], []
    for it in items:
        y_true.append(to_binary_vec(extract_findings(it['gt'])))
        y_pred.append(to_binary_vec(extract_findings(it['pred'])))
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    print(f'\n{task_name} (finding extraction from report): macro F1 = {f1_score(y_true, y_pred, average="macro", zero_division=0):.3f}')


Finding classification: macro F1 = 0.218, micro F1 = 0.385
  Mass                           P=0.524  R=0.342  F1=0.414
  Suspicious Calcification       P=0.687  R=0.543  F1=0.606
  Asymmetry                      P=0.053  R=0.013  F1=0.021
  Architectural Distortion       P=0.000  R=0.000  F1=0.000
  Suspicious Lymph Node          P=0.200  R=0.100  F1=0.133
  Other Suspicious Signs         P=1.000  R=0.071  F1=0.133

BI-RADS:  acc=0.766, weighted kappa=0.561  (4000 valid)
Density:  acc=0.845, weighted kappa=0.621  (4000 valid)

plain_report (finding extraction from report): macro F1 = 0.209


------------

## misc

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyBboxPatch
from matplotlib.gridspec import GridSpec
from PIL import Image
import json

# pick a good case — Model C correctly localizes a Mass
preds = [json.loads(l) for l in open(LORA_RUNS / 'preds_model_c.jsonl')]
report_preds = [p for p in preds if p['task'] == 'grounded_report']

# find one with a clean mass prediction
for p in report_preds:
    image_id = Path(p['image']).stem
    if image_id in image_to_boxes and 'Mass' in [b['group'] for b in image_to_boxes[image_id]]:
        if '<box>' in p['pred']:
            sample = p
            break

image_id = Path(sample['image']).stem
img = Image.open(sample['image'])
W, H = img.size

# Build the figure
fig = plt.figure(figsize=(14, 10))
gs = GridSpec(2, 3, height_ratios=[1, 1.2], width_ratios=[1, 1, 1.2], hspace=0.25, wspace=0.15)

# top row: Image with GT box | Image with predicted box | Side-by-side
ax_gt = fig.add_subplot(gs[0, 0])
ax_gt.imshow(img, cmap='gray')
ax_gt.set_title('Ground Truth', fontsize=12, fontweight='bold')
for item in image_to_boxes.get(image_id, []):
    b = item['box']
    x1, y1, x2, y2 = b[0]/1000*W, b[1]/1000*H, b[2]/1000*W, b[3]/1000*H
    ax_gt.add_patch(Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2.5, edgecolor='lime', facecolor='none'))
    ax_gt.text(x1, y1-15, item['group'], color='lime', fontsize=10,
               bbox=dict(facecolor='black', alpha=0.7, pad=2))
ax_gt.axis('off')

ax_pred = fig.add_subplot(gs[0, 1])
ax_pred.imshow(img, cmap='gray')
ax_pred.set_title('Model C Prediction', fontsize=12, fontweight='bold')
# Parse boxes from prediction
boxes = parse_box_with_label(sample['pred'])
for g, b in boxes:
    x1, y1, x2, y2 = b[0]/1000*W, b[1]/1000*H, b[2]/1000*W, b[3]/1000*H
    ax_pred.add_patch(Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2.5, 
                                edgecolor='red', facecolor='none', linestyle='--'))
    ax_pred.text(x1, y1-15, g, color='red', fontsize=10,
                 bbox=dict(facecolor='black', alpha=0.7, pad=2))
ax_pred.axis('off')

# right: prompt + output text
ax_text = fig.add_subplot(gs[0, 2])
ax_text.axis('off')
prompt = sample['messages'][1]['content'][1]['text'] if isinstance(sample['messages'], list) else "Generate a grounded mammography report..."
ax_text.text(0.0, 0.95, 'INPUT PROMPT:', fontsize=10, fontweight='bold', color='#444', transform=ax_text.transAxes)
ax_text.text(0.0, 0.85, prompt[:200] + '...', fontsize=8.5, family='monospace', wrap=True,
             transform=ax_text.transAxes, verticalalignment='top')
ax_text.text(0.0, 0.55, 'MODEL OUTPUT (raw tokens):', fontsize=10, fontweight='bold', color='#444', transform=ax_text.transAxes)
ax_text.text(0.0, 0.45, sample['pred'][:300], fontsize=8.5, family='monospace', wrap=True,
             transform=ax_text.transAxes, verticalalignment='top')

# bottom row: GT text | rendered output
ax_gt_text = fig.add_subplot(gs[1, :])
ax_gt_text.axis('off')
ax_gt_text.text(0.0, 0.85, 'GROUND TRUTH REPORT:', fontsize=11, fontweight='bold', color='lime', transform=ax_gt_text.transAxes)
ax_gt_text.text(0.0, 0.70, sample['gt'][:400], fontsize=10, family='monospace', wrap=True,
                transform=ax_gt_text.transAxes, verticalalignment='top',
                bbox=dict(facecolor='#f0f0f0', edgecolor='gray', pad=8))
ax_gt_text.text(0.0, 0.35, 'MODEL C PREDICTION (rendered):', fontsize=11, fontweight='bold', color='red', transform=ax_gt_text.transAxes)
ax_gt_text.text(0.0, 0.20, sample['pred'][:400], fontsize=10, family='monospace', wrap=True,
                transform=ax_gt_text.transAxes, verticalalignment='top',
                bbox=dict(facecolor='#fff5f5', edgecolor='red', pad=8))

plt.suptitle(f'Case: {image_id}', fontsize=13, fontweight='bold', y=0.995)
plt.savefig('$VINDR_ROOT/figure3_qualitative.png', dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved to figure3_qualitative.png')

In [ ]:
import re
BOX_RE = re.compile(r'<box>\s*(\d+)\s+(\d+)\s+(\d+)\s+(\d+)\s*</box>')
GROUP_KEYWORDS = {
    'Mass': ['mass'],
    'Suspicious Calcification': ['calcification'],
    'Asymmetry': ['asymmetry'],
    'Architectural Distortion': ['architectural distortion', 'distortion'],
    'Suspicious Lymph Node': ['lymph node'],
    'Other Suspicious Signs': ['other suspicious', 'skin thickening', 'skin retraction', 'nipple retraction'],
}
def parse_box_with_label(pred_text):
    results = []
    for sent in re.split(r'\.\s*', pred_text):
        boxes = BOX_RE.findall(sent)
        if not boxes: continue
        sent_l = sent.lower()
        g = next((gr for gr, kws in GROUP_KEYWORDS.items() if any(k in sent_l for k in kws)), 'Unknown')
        for box in boxes:
            results.append((g, [int(x) for x in box]))
    return results

In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image
from pathlib import Path
import json
import textwrap

# ============================================================
# load YOLO
# ============================================================

YOLO_CKPT = RUNS / "mammo_yolo11s_grouped_v2" / "weights" / "best.pt"
yolo_infer = YOLO(str(YOLO_CKPT))
print(f"Loaded YOLO from {YOLO_CKPT}")

YOLO_ID_TO_GROUP = {i: c for i, c in enumerate(GROUPED_CLASSES)}


def yolo_predict_boxes(image_id, conf_threshold=0.25):
    img_path = PNG / f"{image_id}.png"
    results = yolo_infer.predict(str(img_path), conf=conf_threshold, verbose=False)

    out = []

    for r in results:
        for box in r.boxes:
            cls_id = int(box.cls[0].item())
            conf = float(box.conf[0].item())
            xyxy = box.xyxy[0].tolist()
            out.append((YOLO_ID_TO_GROUP.get(cls_id, "Unknown"), xyxy, conf))

    return out


# ============================================================
# helper: draw boxes
# ============================================================

def draw_gt_boxes(ax, image_id, W, H):
    for item in image_to_boxes.get(image_id, []):
        g = item["group"]
        b = item["box"]

        x1, y1, x2, y2 = denorm_box(b, W, H)

        color = GROUP_COLORS.get(g, "white")

        ax.add_patch(
            Rectangle(
                (x1, y1),
                x2 - x1,
                y2 - y1,
                linewidth=2.5,
                edgecolor=color,
                facecolor="none",
            )
        )

        ax.text(
            x1,
            max(0, y1 - 10),
            g,
            color=color,
            fontsize=8,
            bbox=dict(facecolor="black", alpha=0.7, pad=2),
        )


def draw_vlm_boxes(ax, pred_text, W, H, linestyle="--", linewidth=2.5):
    for g, b in parse_box_with_label(pred_text):
        x1, y1, x2, y2 = denorm_box(b, W, H)

        color = GROUP_COLORS.get(g, "white")

        ax.add_patch(
            Rectangle(
                (x1, y1),
                x2 - x1,
                y2 - y1,
                linewidth=linewidth,
                edgecolor=color,
                facecolor="none",
                linestyle=linestyle,
            )
        )

        ax.text(
            x1,
            max(0, y1 - 10),
            g,
            color=color,
            fontsize=8,
            bbox=dict(facecolor="black", alpha=0.7, pad=2),
        )


def draw_yolo_boxes(ax, image_id, conf_threshold=0.25):
    for g, xyxy, conf in yolo_predict_boxes(image_id, conf_threshold=conf_threshold):
        x1, y1, x2, y2 = xyxy

        color = GROUP_COLORS.get(g, "white")

        ax.add_patch(
            Rectangle(
                (x1, y1),
                x2 - x1,
                y2 - y1,
                linewidth=2.5,
                edgecolor=color,
                facecolor="none",
                linestyle=":",
            )
        )

        ax.text(
            x1,
            max(0, y1 - 10),
            f"{g} ({conf:.2f})",
            color=color,
            fontsize=8,
            bbox=dict(facecolor="black", alpha=0.7, pad=2),
        )


# ============================================================
# load predictions
# ============================================================

preds_b = [json.loads(l) for l in open(LORA_RUNS / "preds_grounded_grouped.jsonl")]
preds_c = [json.loads(l) for l in open(LORA_RUNS / "preds_model_c.jsonl")]

report_b = {
    Path(p["image"]).stem: p
    for p in preds_b
    if p.get("task") == "grounded_report"
}

report_c = {
    Path(p["image"]).stem: p
    for p in preds_c
    if p.get("task") == "grounded_report"
}

common_ids = sorted(set(report_b.keys()) & set(report_c.keys()) & set(image_to_boxes.keys()))

print("Common cases with GT boxes:", len(common_ids))


# ============================================================
# four-panel qualitative figure
# ============================================================

def visualize_four_panel(
    image_id,
    conf_threshold=0.25,
    figsize=(22, 8),
    save_path=None,
):
    assert image_id in report_b, f"{image_id} not found in Model B predictions"
    assert image_id in report_c, f"{image_id} not found in Model C predictions"

    img_path = PNG / f"{image_id}.png"
    img = Image.open(img_path).convert("L")
    W, H = img.size

    fig, axes = plt.subplots(1, 4, figsize=figsize)

    titles = [
        f"Ground Truth\n{image_id}",
        "Model B\nGrounded VLM",
        "Model C\nRare-enriched grounded VLM",
        f"YOLO11s v2\nconf ≥ {conf_threshold}",
    ]

    for ax, title in zip(axes, titles):
        ax.imshow(img, cmap="gray")
        ax.set_title(title, fontsize=11, fontweight="bold")
        ax.axis("off")

    draw_gt_boxes(axes[0], image_id, W, H)
    draw_vlm_boxes(axes[1], report_b[image_id]["pred"], W, H, linestyle="--")
    draw_vlm_boxes(axes[2], report_c[image_id]["pred"], W, H, linestyle="-")
    draw_yolo_boxes(axes[3], image_id, conf_threshold=conf_threshold)

    plt.tight_layout()

    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=250, bbox_inches="tight", facecolor="white")
        print("Saved:", save_path)

    plt.show()


# ============================================================
# Show 5 qualitative cases
# ============================================================

shown = 0

for image_id in common_ids:
    gt_groups = [b["group"] for b in image_to_boxes.get(image_id, [])]

    # prefer examples with clinically useful findings
    if any(g in gt_groups for g in ["Mass", "Suspicious Calcification", "Asymmetry", "Architectural Distortion"]):
        print(f"\n=== {image_id} ===")
        print("GT groups:", gt_groups)
        print("Model B:", report_b[image_id]["pred"][:220])
        print("Model C:", report_c[image_id]["pred"][:220])

        visualize_four_panel(
            image_id,
            conf_threshold=0.25,
            save_path=None,
        )

        shown += 1

    if shown >= 5:
        break

Loaded YOLO from $VINDR_ROOT/runs/mammo_yolo11s_grouped_v2/weights/best.pt
Common cases with GT boxes: 357

=== <image_id> ===
GT groups: ['Suspicious Calcification']
Model B: The BI-RADS category is BI-RADS 2. The breast density is DENSITY C. No suspicious mammographic finding is present.
Model C: The BI-RADS category is BI-RADS 3. The breast density is DENSITY C. No suspicious mammographic finding is present.


In [ ]:
def visualize_four_panel_with_text(
    image_id,
    conf_threshold=0.25,
    figsize=(22, 13),
    save_path=None,
):
    assert image_id in report_b, f"{image_id} not found in Model B predictions"
    assert image_id in report_c, f"{image_id} not found in Model C predictions"

    img_path = PNG / f"{image_id}.png"
    img = Image.open(img_path).convert("L")
    W, H = img.size

    fig = plt.figure(figsize=figsize)
    gs = fig.add_gridspec(2, 4, height_ratios=[1.0, 0.55], hspace=0.15, wspace=0.08)

    axes = [fig.add_subplot(gs[0, i]) for i in range(4)]

    titles = [
        f"Ground Truth\n{image_id}",
        "Model B\nGrounded VLM",
        "Model C\nRare-enriched grounded VLM",
        f"YOLO11s v2\nconf ≥ {conf_threshold}",
    ]

    for ax, title in zip(axes, titles):
        ax.imshow(img, cmap="gray")
        ax.set_title(title, fontsize=11, fontweight="bold")
        ax.axis("off")

    draw_gt_boxes(axes[0], image_id, W, H)
    draw_vlm_boxes(axes[1], report_b[image_id]["pred"], W, H, linestyle="--")
    draw_vlm_boxes(axes[2], report_c[image_id]["pred"], W, H, linestyle="-")
    draw_yolo_boxes(axes[3], image_id, conf_threshold=conf_threshold)

    ax_text = fig.add_subplot(gs[1, :])
    ax_text.axis("off")

    gt_text = report_c[image_id].get("gt", report_b[image_id].get("gt", ""))
    b_text = report_b[image_id]["pred"]
    c_text = report_c[image_id]["pred"]

    def wrap(s, width=115):
        s = str(s).replace("\n", " ")
        return "\n".join(textwrap.wrap(s, width=width))

    text_block = (
        "GROUND TRUTH:\n"
        + wrap(gt_text[:500])
        + "\n\nMODEL B OUTPUT:\n"
        + wrap(b_text[:500])
        + "\n\nMODEL C OUTPUT:\n"
        + wrap(c_text[:500])
    )

    ax_text.text(
        0.01,
        0.98,
        text_block,
        transform=ax_text.transAxes,
        va="top",
        ha="left",
        fontsize=9,
        family="monospace",
        bbox=dict(facecolor="#f7f7f7", edgecolor="gray", alpha=0.95, pad=8),
    )

    plt.suptitle(f"Qualitative grounding comparison: {image_id}", fontsize=14, fontweight="bold", y=0.99)

    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=250, bbox_inches="tight", facecolor="white")
        print("Saved:", save_path)

    plt.show()

In [ ]:
from pathlib import Path

DATA_ROOT = Path(os.environ.get('VINDR_ROOT', 'data/vindr_mammo'))  # see README -> Data

# OLD working grouped-model folder
LORA_RUNS = DATA_ROOT / 'lora_runs'

print("LORA_RUNS =", LORA_RUNS)
print("exists?", LORA_RUNS.exists())
print("files:")
for p in LORA_RUNS.glob("preds*.jsonl"):
    print(" ", p.name)

LORA_RUNS = $VINDR_ROOT/lora_runs
exists? True
files:
  preds_grounded_grouped.jsonl
  preds_model_c.jsonl
  preds_zero_shot.jsonl
  preds_plain_grouped.jsonl


In [ ]:
import json, re
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support

GROUPED_CLASSES = [
    'Mass', 'Suspicious Calcification', 'Asymmetry',
    'Architectural Distortion', 'Suspicious Lymph Node', 'Other Suspicious Signs'
]

GROUP_KEYWORDS = {
    'Mass': ['mass'],
    'Suspicious Calcification': ['calcification', 'calcifications', 'calcified'],
    'Asymmetry': ['asymmetry'],
    'Architectural Distortion': ['architectural distortion', 'distortion'],
    'Suspicious Lymph Node': ['lymph node'],
    'Other Suspicious Signs': ['other suspicious', 'skin thickening', 'skin retraction', 'nipple retraction'],
}

BOX_RE = re.compile(r'<box>\s*(\d+)\s+(\d+)\s+(\d+)\s+(\d+)\s*</box>', re.I)

NEGATIVE_PHRASES = [
    'no suspicious mammographic finding',
    'no suspicious finding',
    'no finding',
    'no suspicious abnormality',
    'no mammographic evidence of malignancy',
]

def extract_findings_simple(text):
    t = str(text).lower()

    # If the model explicitly says no suspicious finding, treat as no finding
    if any(p in t for p in NEGATIVE_PHRASES):
        return set()

    out = set()
    for g, kws in GROUP_KEYWORDS.items():
        if any(k in t for k in kws):
            out.add(g)
    return out

def has_box(text):
    return len(BOX_RE.findall(str(text))) > 0

def binary_from_grouped_text_or_box(text):
    groups = extract_findings_simple(text)
    return 1 if (len(groups) > 0 or has_box(text)) else 0

PRED_FILES = {
    'Model A old grouped': LORA_RUNS / 'preds_plain_grouped.jsonl',
    'Model B old grouped': LORA_RUNS / 'preds_grounded_grouped.jsonl',
    'Model C old grouped': LORA_RUNS / 'preds_model_c.jsonl',
}

rows = []

for model_name, path in PRED_FILES.items():
    print("\n" + "="*80)
    print(model_name, path)

    if not path.exists():
        print("FILE NOT FOUND")
        continue

    preds = [json.loads(l) for l in open(path)]

    for task_name in ['finding_classification', 'plain_report', 'grounded_report']:
        items = [p for p in preds if p.get('task') == task_name]

        if len(items) == 0:
            continue

        y_true = np.array([binary_from_grouped_text_or_box(it['gt']) for it in items])
        y_pred = np.array([binary_from_grouped_text_or_box(it['pred']) for it in items])

        acc = accuracy_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        ppv, sens, _, _ = precision_recall_fscore_support(
            y_true, y_pred, average='binary', zero_division=0
        )

        tn = int(((y_true == 0) & (y_pred == 0)).sum())
        fp = int(((y_true == 0) & (y_pred == 1)).sum())
        fn = int(((y_true == 1) & (y_pred == 0)).sum())
        tp = int(((y_true == 1) & (y_pred == 1)).sum())

        spec = tn / (tn + fp) if (tn + fp) else 0
        npv = tn / (tn + fn) if (tn + fn) else 0

        print(f"\n{task_name}")
        print(f"  n={len(items)} positives={int(y_true.sum())} predicted_pos={int(y_pred.sum())}")
        print(f"  Acc={acc:.3f} F1={f1:.3f} PPV={ppv:.3f} Sens={sens:.3f} Spec={spec:.3f} NPV={npv:.3f}")
        print(f"  TP={tp} FP={fp} FN={fn} TN={tn}")

        rows.append({
            'model': model_name,
            'task': task_name,
            'n': len(items),
            'gt_pos': int(y_true.sum()),
            'pred_pos': int(y_pred.sum()),
            'acc': acc,
            'f1': f1,
            'ppv': ppv,
            'sensitivity': sens,
            'specificity': spec,
            'npv': npv,
            'tp': tp,
            'fp': fp,
            'fn': fn,
            'tn': tn,
        })

binary_df = pd.DataFrame(rows)
display(binary_df)
binary_df.to_csv(LORA_RUNS / 'old_grouped_as_binary_suspicious.csv', index=False)
print("Saved:", LORA_RUNS / 'old_grouped_as_binary_suspicious.csv')


Model A old grouped $VINDR_ROOT/lora_runs/preds_plain_grouped.jsonl

finding_classification
  n=4000 positives=357 predicted_pos=188
  Acc=0.925 F1=0.451 PPV=0.654 Sens=0.345 Spec=0.982 NPV=0.939
  TP=123 FP=65 FN=234 TN=3578

plain_report
  n=4000 positives=357 predicted_pos=196
  Acc=0.924 F1=0.448 PPV=0.633 Sens=0.347 Spec=0.980 NPV=0.939
  TP=124 FP=72 FN=233 TN=3571

Model B old grouped $VINDR_ROOT/lora_runs/preds_grounded_grouped.jsonl

finding_classification
  n=4000 positives=357 predicted_pos=194
  Acc=0.931 F1=0.497 PPV=0.706 Sens=0.384 Spec=0.984 NPV=0.942
  TP=137 FP=57 FN=220 TN=3586

grounded_report
  n=4000 positives=357 predicted_pos=215
  Acc=0.925 F1=0.476 PPV=0.633 Sens=0.381 Spec=0.978 NPV=0.942
  TP=136 FP=79 FN=221 TN=3564

Model C old grouped $VINDR_ROOT/lora_runs/preds_model_c.jsonl

finding_classification
  n=4000 positives=357 predicted_pos=528
  Acc=0.881 F1=0.461 PPV=0.386 Sens=0.571 Spec=0.911 NPV=0.956
  TP=204 FP=324 FN=153 TN=3319

grounded_report
  n=4

,model,task,n,gt_pos,pred_pos,acc,f1,ppv,sensitivity,specificity,npv,tp,fp,fn,tn
0,Model A old grouped,finding_classification,4000,357,188,0.92525,0.451376,0.654255,0.344538,0.982158,0.938615,123,65,234,3578
1,Model A old grouped,plain_report,4000,357,196,0.92375,0.448463,0.632653,0.347339,0.980236,0.938749,124,72,233,3571
2,Model B old grouped,finding_classification,4000,357,194,0.93075,0.497278,0.706186,0.383754,0.984354,0.942197,137,57,220,3586
3,Model B old grouped,grounded_report,4000,357,215,0.92500,0.475524,0.632558,0.380952,0.978315,0.941612,136,79,221,3564
4,Model C old grouped,finding_classification,4000,357,528,0.88075,0.461017,0.386364,0.571429,0.911062,0.955933,204,324,153,3319
5,Model C old grouped,grounded_report,4000,357,616,0.86125,0.429599,0.339286,0.585434,0.888279,0.956265,209,407,148,3236


Saved: $VINDR_ROOT/lora_runs/old_grouped_as_binary_suspicious.csv
